In [1]:
# CÉLULA 1: Classes, Herança e Validações de Negócio

class Roupa:
    # Lista de siglas padronizadas aceitas pelo sistema
    TAMANHOS_VALIDOS = ['PP', 'P', 'M', 'G', 'GG', 'XG', 'UNICO']

    def __init__(self, codigo: int, descricao: str, categoria: str, tamanho: str, cor: str, preco: float, quantidade: int):
        self.codigo = int(codigo)
        self.descricao = str(descricao).strip()
        self.categoria = str(categoria).strip()
        self.cor = str(cor).strip()

        # Validações obrigatórias disparadas no construtor
        self.tamanho = self._validar_tamanho(tamanho)
        self.preco = self._validar_preco(preco)
        self.quantidade = self._validar_quantidade(quantidade)

    def _validar_tamanho(self, tamanho: str) -> str:
        tam_upper = str(tamanho).strip().upper()
        if tam_upper not in self.TAMANHOS_VALIDOS:
            # Permite numerações de calça/vestido (ex: 38, 40, 42) se forem estritamente numéricas
            if not tam_upper.isdigit():
                raise ValueError(f"Tamanho inválido. Escolha entre {self.TAMANHOS_VALIDOS} ou uma numeração válida.")
        return tam_upper

    def _validar_preco(self, preco: float) -> float:
        val = float(preco)
        if val <= 0:
            raise ValueError("O preço de venda deve ser estritamente positivo (maior que zero).")
        return val

    def _validar_quantidade(self, quantidade: int) -> int:
        val = int(quantidade)
        if val < 0:
            raise ValueError("A quantidade em estoque não pode ser negativa.")
        return val

    def editar_dados(self, descricao=None, tamanho=None, cor=None, preco=None, quantity=None):
        """Altera os atributos do objeto aplicando as regras de validação."""
        if descricao and str(descricao).strip():
            self.descricao = str(descricao).strip()
        if cor and str(cor).strip():
            self.cor = str(cor).strip()
        if tamanho and str(tamanho).strip():
            self.tamanho = self._validar_tamanho(tamanho)
        if preco is not None:
            self.preco = self._validar_preco(preco)
        if quantity is not None:
            self.quantidade = self._validar_quantidade(quantity)

    def movimentar_estoque(self, quantidade_venda: int):
        """Realiza a baixa do estoque validando o saldo disponível."""
        if quantidade_venda <= 0:
            raise ValueError("A quantidade vendida deve ser maior que zero.")
        if self.quantidade - quantidade_venda < 0:
            raise ValueError(f"Estoque insuficiente. Saldo atual: {self.quantidade} unidade(s).")
        self.quantidade -= quantidade_venda

    def to_dict(self) -> dict:
        """Serializa o objeto para ser armazenado em arquivos JSON."""
        return {
            "tipo_classe": self.__class__.__name__,
            "codigo": self.codigo,
            "descricao": self.descricao,
            "categoria": self.categoria,
            "tamanho": self.tamanho,
            "cor": self.cor,
            "preco": self.preco,
            "quantidade": self.quantidade
        }


# Subclasses aplicando o conceito de Herança
class RoupaCasual(Roupa):
    def __init__(self, codigo, descricao, tamanho, cor, preco, quantidade):
        super().__init__(codigo, descricao, "Casual", tamanho, cor, preco, quantidade)

class RoupaFormal(Roupa):
    def __init__(self, codigo, descricao, tamanho, cor, preco, quantidade):
        super().__init__(codigo, descricao, "Formal", tamanho, cor, preco, quantidade)

class RoupaEsportiva(Roupa):
    def __init__(self, codigo, descricao, tamanho, cor, preco, quantidade):
        super().__init__(codigo, descricao, "Esportiva", tamanho, cor, preco, quantidade)

print("✔️ Célula 1 executada: Classes de POO prontas.")

✔️ Célula 1 executada: Classes de POO prontas.


In [2]:
# CÉLULA 2: Estruturas de Dados e Mecanismos de Busca

class Estoque:
    def __init__(self):
        # Estruturas obrigatórias solicitadas pelo enunciado
        self.vetor_nao_ordenado = []
        self.vetor_ordenado = []

    def cadastrar_roupa(self, roupa: Roupa):
        """Insere uma peça garantindo a integridade e sincronia de ambos os vetores."""
        if self._busca_binaria_indice(roupa.codigo) != -1:
            raise ValueError(f"Já existe uma peça cadastrada com o código {roupa.codigo}.")

        # 1. Vetor Não Ordenado: Inserção rápida O(1)
        self.vetor_nao_ordenado.append(roupa)

        # 2. Vetor Ordenado: Inserção mantendo o vetor ordenado O(n)
        self._inserir_mantendo_ordenacao(roupa)

    def _inserir_mantendo_ordenacao(self, roupa: Roupa):
        """Algoritmo posicional estável baseado no Insertion Sort."""
        i = 0
        while i < len(self.vetor_ordenado) and self.vetor_ordenado[i].codigo < roupa.codigo:
            i += 1
        self.vetor_ordenado.insert(i, roupa)

    def _busca_binaria_indice(self, codigo: int) -> int:
        """Algoritmo de Busca Binária. Retorna a posição (índice) exata no vetor. Complexidade O(log n)."""
        arr = self.vetor_ordenado
        baixa = 0
        alta = len(arr) - 1

        while baixa <= alta:
            meio = (baixa + alta) // 2
            if arr[meio].codigo == codigo:
                return meio
            elif arr[meio].codigo < codigo:
                baixa = meio + 1
            else:
                alta = meio - 1
        return -1

    def buscar_por_codigo(self, codigo: int) -> Roupa:
        """Interface pública para obter o objeto através de Busca Binária."""
        idx = self._busca_binaria_indice(codigo)
        if idx != -1:
            return self.vetor_ordenado[idx]
        return None

    def buscar_por_descricao(self, termo: str) -> list:
        """Algoritmo de Busca Linear no Vetor Não Ordenado. Complexidade O(n)."""
        resultados = []
        termo_lower = str(termo).lower().strip()
        for roupa in self.vetor_nao_ordenado:
            if termo_lower in roupa.descricao.lower():
                resultados.append(roupa)
        return resultados

    def remover_por_codigo(self, codigo: int) -> bool:
        """Remoção otimizada eliminando buscas lineares internas duplicadas."""
        idx_ordenado = self._busca_binaria_indice(codigo)
        if idx_ordenado != -1:
            roupa = self.vetor_ordenado[idx_ordenado]
            # Remove usando a posição direta da busca binária
            self.vetor_ordenado.pop(idx_ordenado)
            self.vetor_nao_ordenado.remove(roupa)
            return True
        return False

    def filtrar_por_categoria(self, categoria: str) -> list:
        return [r for r in self.vetor_nao_ordenado if r.categoria.lower() == categoria.lower().strip()]

    def filtrar_por_tamanho(self, tamanho: str) -> list:
        return [r for r in self.vetor_nao_ordenado if r.tamanho.upper() == tamanho.upper().strip()]

    def relatorio_estoque_baixo(self, limite: int) -> list:
        return [r for r in self.vetor_nao_ordenado if r.quantidade < limite]

print("✔️ Célula 2 executada: Estrutura do Estoque e Algoritmos Big-O prontos.")

✔️ Célula 2 executada: Estrutura do Estoque e Algoritmos Big-O prontos.


In [3]:
# CÉLULA 3: Persistência e Recuperação de Arquivos JSON
import json
import os

MAPA_CLASSES = {
    "Roupa": Roupa,
    "RoupaCasual": RoupaCasual,
    "RoupaFormal": RoupaFormal,
    "RoupaEsportiva": RoupaEsportiva
}

def salvar_dados(estoque: Estoque, filename="estoque_roupas.json"):
    """Salva os dados do estoque no disco da máquina virtual do Colab."""
    dados = [roupa.to_dict() for roupa in estoque.vetor_nao_ordenado]
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(dados, f, indent=4, ensure_ascii=False)

def carregar_dados(filename="estoque_roupas.json") -> Estoque:
    """Carrega os dados reconstruindo as instâncias corretas de POO."""
    estoque = Estoque()
    if not os.path.exists(filename):
        return estoque

    try:
        with open(filename, 'r', encoding='utf-8') as f:
            dados = json.load(f)
            for item in dados:
                tipo_classe = item.get("tipo_classe", "Roupa")
                classe_alvo = MAPA_CLASSES.get(tipo_classe, Roupa)

                if classe_alvo == Roupa:
                    roupa = classe_alvo(
                        codigo=item["codigo"], descricao=item["descricao"], categoria=item["categoria"],
                        tamanho=item["tamanho"], cor=item["cor"], preco=item["preco"], quantidade=item["quantidade"]
                    )
                else:
                    # Subclasses cuidam da categoria sozinhas em seus construtores
                    roupa = classe_alvo(
                        codigo=item["codigo"], descricao=item["descricao"],
                        tamanho=item["tamanho"], cor=item["cor"], preco=item["preco"], quantidade=item["quantidade"]
                    )
                estoque.cadastrar_roupa(roupa)
    except (json.JSONDecodeError, KeyError, FileNotFoundError):
        print("⚠️ Aviso: Arquivo JSON vazio ou corrompido. Iniciando um estoque novo e limpo.")

    return estoque

print("✔️ Célula 3 executada: Módulo de arquivos integrado.")

✔️ Célula 3 executada: Módulo de arquivos integrado.


In [4]:
# CÉLULA 4: Menu Interativo e Interface CLI

def exibir_tabela_paginada(lista_roupas, itens_por_pagina=5):
    """Exibe os dados formatados em formato de tabela com paginação simples."""
    if not lista_roupas:
        print("\n[ NENHUM PRODUTO ENCONTRADO PARA ESTA OPERAÇÃO ]")
        return

    total = len(lista_roupas)
    for i in range(0, total, itens_por_pagina):
        pagina = lista_roupas[i:i+itens_por_pagina]
        print(f"\n=================== EXIBINDO ITENS {i+1} - {min(i+itens_por_pagina, total)} DE {total} ===================")
        print(f"{'CÓD':<6} | {'DESCRIÇÃO':<25} | {'CATEGORIA':<12} | {'TAM':<5} | {'COR':<10} | {'PREÇO':<10} | {'QTD':<5}")
        print("-" * 85)
        for r in pagina:
            print(f"{r.codigo:<6} | {r.descricao:<25} | {r.categoria:<12} | {r.tamanho:<5} | {r.cor:<10} | R$ {r.preco:<7.2f} | {r.quantidade:<5}")

        if i + itens_por_pagina < total:
            opcao = input("\nPressione [Enter] para a próxima página ou [M] para interromper: ").strip().lower()
            if opcao == 'm':
                break

def ler_inteiro(mensagem: str) -> int:
    while True:
        try:
            return int(input(mensagem))
        except ValueError:
            print("❌ Entrada inválida. Por favor, digite um número inteiro.")

def ler_float(mensagem: str) -> float:
    while True:
        try:
            return float(input(mensagem).replace(",", "."))
        except ValueError:
            print("❌ Entrada inválida. Por favor, digite um valor monetário válido (ex: 79.90).")

def menu_principal():
    estoque = carregar_dados()

    while True:
        print("\n" + "="*45)
        print("     SISTEMA BOUTIQUE - GESTÃO DE ESTOQUE     ")
        print("="*45)
        print("1. Cadastrar Nova Peça de Roupa")
        print("2. Editar Dados de uma Peça")
        print("3. Remover Peça do Sistema")
        print("4. Buscar por Código (Busca Binária)")
        print("5. Buscar por Nome/Descrição (Busca Linear)")
        print("6. Registrar Venda de Peça")
        print("7. Listar Estoque Geral (Ordenado por Código)")
        print("8. Filtrar Roupas (Por Categoria ou Tamanho)")
        print("9. Relatório de Alerta de Estoque Baixo")
        print("10. Salvar Dados e Encerrar Sistema")
        print("="*45)

        opcao = input("Selecione a operação desejada (1-10): ").strip()

        if opcao == "1":
            print("\n--- CADASTRO DE PRODUTO ---")
            print("Escolha o Estilo:\n1. Casual\n2. Formal\n3. Esportiva\n4. Outro Estilo")
            tipo = input("Opção: ").strip()

            try:
                cod = ler_inteiro("Código Único Numérico: ")
                desc = input("Descrição da Peça (ex: Vestido Midi): ").strip()
                tam = input("Tamanho (PP, P, M, G, GG, 38, 40): ").strip()
                cor = input("Cor principal: ").strip()
                preco = ler_float("Preço de Venda: R$ ")
                qtd = ler_inteiro("Quantidade inicial em estoque: ")

                if not desc or not cor:
                    print("❌ Erro: Campos de texto não podem ser nulos.")
                    continue

                if tipo == "1":
                    nova_roupa = RoupaCasual(cod, desc, tam, cor, preco, qtd)
                elif tipo == "2":
                    nova_roupa = RoupaFormal(cod, desc, tam, cor, preco, qtd)
                elif tipo == "3":
                    nova_roupa = RoupaEsportiva(cod, desc, tam, cor, preco, qtd)
                else:
                    cat = input("Digite o nome da categoria customizada: ").strip()
                    if not cat: cat = "Geral"
                    nova_roupa = Roupa(cod, desc, cat, tam, cor, preco, qtd)

                estoque.cadastrar_roupa(nova_roupa)
                print("✔️ Produto adicionado com sucesso ao inventário!")
            except ValueError as e:
                print(f"❌ Falha de validação: {e}")

        elif opcao == "2":
            print("\n--- ALTERAÇÃO DE DADOS ---")
            cod = ler_inteiro("Insira o código do item que deseja editar: ")
            roupa = estoque.buscar_por_codigo(cod)
            if roupa:
                print(f"Modificando item: {roupa.descricao} ({roupa.categoria})")
                print("(Deixe em branco e aperte [Enter] para não alterar o atributo)")

                desc = input(f"Nova descrição [{roupa.descricao}]: ").strip()
                cor = input(f"Nova cor [{roupa.cor}]: ").strip()
                tam = input(f"Novo tamanho [{roupa.tamanho}]: ").strip()

                p_in = input(f"Novo preço [{roupa.preco}]: ").strip()
                preco = float(p_in.replace(",", ".")) if p_in else None

                q_in = input(f"Nova quantidade [{roupa.quantidade}]: ").strip()
                qtd = int(q_in) if q_in else None

                try:
                    roupa.editar_dados(descricao=desc, tamanho=tam, cor=cor, preco=preco, quantity=qtd)
                    print("✔️ Dados salvos e alterados no objeto com sucesso!")
                except ValueError as e:
                    print(f"❌ Erro na modificação: {e}")
            else:
                print("❌ Produto não localizado.")

        elif opcao == "3":
            print("\n--- REMOVER PRODUTO ---")
            cod = ler_inteiro("Código do item a ser excluído definitivamente: ")
            if estoque.remover_por_codigo(cod):
                print("✔️ Registro retirado com sucesso de todas as tabelas.")
            else:
                print("❌ Código inexistente.")

        elif opcao == "4":
            print("\n--- CONSULTA POR CÓDIGO (BUSCA BINÁRIA) ---")
            cod = ler_inteiro("Digite o código numérico procurado: ")
            roupa = estoque.buscar_por_codigo(cod)
            if roupa:
                print(f"\n🔍 Localizado com Busca Binária:")
                print(f"Código: {roupa.codigo} | Peça: {roupa.descricao} | Preço: R$ {roupa.preco:.2f} | Saldo: {roupa.quantidade}")
            else:
                print("❌ Nenhuma peça corresponde ao código digitado.")

        elif opcao == "5":
            print("\n--- BUSCA POR NOME (BUSCA LINEAR) ---")
            termo = input("Digite o nome ou parte da descrição da roupa: ")
            resultados = estoque.buscar_por_descricao(termo)
            exibir_tabela_paginada(resultados)

        elif opcao == "6":
            print("\n--- REGISTRAR MOVIMENTAÇÃO DE VENDA ---")
            cod = ler_inteiro("Código do item vendido: ")
            roupa = estoque.buscar_por_codigo(cod)
            if roupa:
                qtd_venda = ler_inteiro(f"Quantidade vendida (Em estoque: {roupa.quantidade}): ")
                try:
                    roupa.movimentar_estoque(qtd_venda)
                    print(f"✔️ Venda registrada com sucesso! Novo saldo de '{roupa.descricao}': {roupa.quantidade} un.")
                except ValueError as e:
                    print(f"❌ Operação cancelada: {e}")
            else:
                print("❌ Produto não cadastrado.")

        elif opcao == "7":
            print("\n--- INVENTÁRIO COMPLETO (ORDENADO POR CÓDIGO) ---")
            exibir_tabela_paginada(estoque.vetor_ordenado)

        elif opcao == "8":
            print("\n--- FILTRAR CATÁLOGO ---")
            print("1. Filtrar por Tipo/Categoria\n2. Filtrar por Tamanho específico")
            sub = input("Escolha o filtro: ").strip()
            if sub == "1":
                cat = input("Nome da categoria procurada: ")
                exibir_tabela_paginada(estoque.filtrar_por_categoria(cat))
            elif sub == "2":
                tam = input("Tamanho procurado: ")
                exibir_tabela_paginada(estoque.filtrar_por_tamanho(tam))
            else:
                print("Opção inválida.")

        elif opcao == "9":
            print("\n--- RELATÓRIO CRÍTICO: ESTOQUE BAIXO ---")
            limite = ler_inteiro("Defina a quantidade limite para gerar o alerta: ")
            resultados = estoque.relatorio_estoque_baixo(limite)
            exibir_tabela_paginada(resultados)

        elif opcao == "10":
            salvar_dados(estoque)
            print("\n💾 Base de dados salva na nuvem com sucesso.")
            print("Sistema encerrado de maneira segura no ambiente Colab.")
            break
        else:
            print("❌ Escolha inválida. Selecione uma opção de 1 a 10.")

# Inicializador automático do menu
menu_principal()


     SISTEMA BOUTIQUE - GESTÃO DE ESTOQUE     
1. Cadastrar Nova Peça de Roupa
2. Editar Dados de uma Peça
3. Remover Peça do Sistema
4. Buscar por Código (Busca Binária)
5. Buscar por Nome/Descrição (Busca Linear)
6. Registrar Venda de Peça
7. Listar Estoque Geral (Ordenado por Código)
8. Filtrar Roupas (Por Categoria ou Tamanho)
9. Relatório de Alerta de Estoque Baixo
10. Salvar Dados e Encerrar Sistema
Selecione a operação desejada (1-10): 1

--- CADASTRO DE PRODUTO ---
Escolha o Estilo:
1. Casual
2. Formal
3. Esportiva
4. Outro Estilo
Opção: 1
Código Único Numérico: 123
Descrição da Peça (ex: Vestido Midi): Vestido
Tamanho (PP, P, M, G, GG, 38, 40): pp
Cor principal: azul
Preço de Venda: R$ 1000
Quantidade inicial em estoque: 123
✔️ Produto adicionado com sucesso ao inventário!

     SISTEMA BOUTIQUE - GESTÃO DE ESTOQUE     
1. Cadastrar Nova Peça de Roupa
2. Editar Dados de uma Peça
3. Remover Peça do Sistema
4. Buscar por Código (Busca Binária)
5. Buscar por Nome/Descrição (Busc